# NOT training: MNIST red even → blue odd

Ported from the paper experiment (`mnist_red_even_to_blue_odd/train_mnist.py`): source = all even MNIST digits rendered in red, target = all odd MNIST digits rendered in blue, trained with the repo's generic `train_gdmax` / `train_extragradient`.

**How to run.** Run the setup cells (imports, *Data*, *Helpers*), then any training section on its own: each one seeds with `SEED`, builds a fresh `T` and `f`, reads its step sizes from `LR`, and is followed by a plotting cell.

In [ ]:
import os, sys
sys.path.append("..")

import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import clear_output

device = "cuda" if torch.cuda.is_available() else "cpu"

from src.models import TransportUNet, PotentialResNet, warmup_spectral_norm
from src.datasets import build_mnist_red_even_blue_odd, LoaderSampler
from src.utils import set_seed, cosine_decay, plot_loss, plot_grad_norm, show_image_samples
from src.train import train_gdmax, train_extragradient, train_ogda

SEED = 0
IMG_SIZE = 32
BATCH_SIZE = 120
N_STEPS = 50_000

EXP_DIR = "../outputs/mnist_red_even_to_blue_odd"
# Step sizes (gamma_T, gamma_V) of each training section.
LR = {
    "gdmax": (1e-4, 1e-4),
    "eg": (1e-4, 1e-4),
    "tts_semi_dual": (2e-4, 1e-4),
    "tts_lagrangian": (1e-4, 2e-4),
    "ogda": (1e-4, 1e-4),
}


## Data: red even MNIST → blue odd MNIST

Train and test splits are combined (no train/test split), matching the paper setup.

In [ ]:
source_data, target_data = build_mnist_red_even_blue_odd("../data", img_size=IMG_SIZE)
print(f"red even samples: {len(source_data):,}")
print(f"blue odd samples: {len(target_data):,}")

source_sampler = LoaderSampler(source_data, batch_size=BATCH_SIZE, seed=SEED, device=device)
target_sampler = LoaderSampler(target_data, batch_size=BATCH_SIZE, seed=SEED + 1, device=device)


def sample_mu(batch_size):
    return source_sampler.sample()


def sample_nu(batch_size):
    return target_sampler.sample()


# Fixed images so every saved figure compares the same source/target samples.
fixed_x_sampler = LoaderSampler(source_data, batch_size=10, seed=SEED + 10_000, device=device)
fixed_y_sampler = LoaderSampler(target_data, batch_size=10, seed=SEED + 10_001, device=device)
X_FIXED = fixed_x_sampler.sample()
Y_FIXED = fixed_y_sampler.sample()

cost = F.mse_loss
ADAM_KWARGS = dict(betas=(0.0, 0.9), weight_decay=1e-10, amsgrad=True)
GRAD_CLIP = 10.0


## Helpers

Every training section below starts from `build_models()` (seeded with `SEED`) and writes to `EXP_DIR/<method>`.

In [ ]:
def run_dir(key):
    path = os.path.join(EXP_DIR, key)
    os.makedirs(path, exist_ok=True)
    return path


def build_models():
    """Seed, then a fresh T and spectral-normalized f; every training section starts here."""
    set_seed(SEED)
    T = TransportUNet(base_channels=32).to(device)
    f = PotentialResNet(base_channels=32).to(device)
    warmup_spectral_norm(f, IMG_SIZE, device=device)
    return T, f


def schedule(lr_T, lr_f):
    return lambda step: (cosine_decay(step, N_STEPS, lr_T), cosine_decay(step, N_STEPS, lr_f))


def sample_callback(key):
    return lambda step, T, f: show_image_samples(
        T, X_FIXED, Y_FIXED, method_name=key,
        save_path=os.path.join(run_dir(key), f"samples_step_{step + 1:06d}.png"))


def finish(key, T, history):
    plot_loss(history["loss"], save_path=os.path.join(run_dir(key), "loss.png"))
    plot_grad_norm(history["grad_T"], history["grad_f"], save_path=os.path.join(run_dir(key), "grad_norm.png"))
    show_image_samples(T, X_FIXED, Y_FIXED, method_name=key,
                       save_path=os.path.join(run_dir(key), "final_samples.png"))


## GDmax

$K = 5$ Adam descent steps on `T` per Adam ascent step on `f`, each on a fresh batch (`train_gdmax`).

In [ ]:
T_gdmax, f_gdmax = build_models()
lr_T, lr_f = LR["gdmax"]

history_gdmax = train_gdmax(
    T_gdmax, f_gdmax, sample_mu, sample_nu, cost,
    n_steps=N_STEPS, lr_T=lr_T, lr_f=lr_f, K=5,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    lr_schedule=schedule(lr_T, lr_f), grad_clip=GRAD_CLIP, optimizer_kwargs=ADAM_KWARGS,
    log_dir=run_dir("gdmax"), callback=sample_callback("gdmax"),
)


In [ ]:
finish("gdmax", T_gdmax, history_gdmax)


## EG

Extragradient through Adam with equal step sizes (`train_extragradient(use_adam=True)`): the look-ahead and the update use one batch, and the Adam state is restored before the update.

In [ ]:
T_eg, f_eg = build_models()
lr_T, lr_f = LR["eg"]

history_eg = train_extragradient(
    T_eg, f_eg, sample_mu, sample_nu, cost,
    n_steps=N_STEPS, lr_T=lr_T, lr_f=lr_f,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    lr_schedule=schedule(lr_T, lr_f), grad_clip=GRAD_CLIP, use_adam=True, optimizer_kwargs=ADAM_KWARGS,
    log_dir=run_dir("eg"), callback=sample_callback("eg"),
)


In [ ]:
finish("eg", T_eg, history_eg)


## TTS-EG (Semi-Dual)

Two-timescale extragradient for $\max_V \min_T$: $\gamma_T > \gamma_V$.

In [ ]:
T_tts_semi_dual, f_tts_semi_dual = build_models()
lr_T, lr_f = LR["tts_semi_dual"]

history_tts_semi_dual = train_extragradient(
    T_tts_semi_dual, f_tts_semi_dual, sample_mu, sample_nu, cost,
    n_steps=N_STEPS, lr_T=lr_T, lr_f=lr_f,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    lr_schedule=schedule(lr_T, lr_f), grad_clip=GRAD_CLIP, use_adam=True, optimizer_kwargs=ADAM_KWARGS,
    log_dir=run_dir("tts_semi_dual"), callback=sample_callback("tts_semi_dual"),
)


In [ ]:
finish("tts_semi_dual", T_tts_semi_dual, history_tts_semi_dual)


## TTS-EG (Lagrangian)

Two-timescale extragradient for $\min_T \max_V$: $\gamma_T < \gamma_V$.

In [ ]:
T_tts_lagrangian, f_tts_lagrangian = build_models()
lr_T, lr_f = LR["tts_lagrangian"]

history_tts_lagrangian = train_extragradient(
    T_tts_lagrangian, f_tts_lagrangian, sample_mu, sample_nu, cost,
    n_steps=N_STEPS, lr_T=lr_T, lr_f=lr_f,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    lr_schedule=schedule(lr_T, lr_f), grad_clip=GRAD_CLIP, use_adam=True, optimizer_kwargs=ADAM_KWARGS,
    log_dir=run_dir("tts_lagrangian"), callback=sample_callback("tts_lagrangian"),
)


In [ ]:
finish("tts_lagrangian", T_tts_lagrangian, history_tts_lagrangian)


## OGDA

Optimistic Adam (`train_ogda`, `src/optim.py`) with the Adam settings above ($\beta_1 = 0$), one gradient evaluation per step.

In [ ]:
T_ogda, f_ogda = build_models()
lr_T, lr_f = LR["ogda"]

history_ogda = train_ogda(
    T_ogda, f_ogda, sample_mu, sample_nu, cost,
    n_steps=N_STEPS, lr_T=lr_T, lr_f=lr_f,
    batch_size_x=BATCH_SIZE, batch_size_y=BATCH_SIZE,
    lr_schedule=schedule(lr_T, lr_f), grad_clip=GRAD_CLIP, optimizer_kwargs=ADAM_KWARGS,
    log_dir=run_dir("ogda"), callback=sample_callback("ogda"),
)


In [ ]:
finish("ogda", T_ogda, history_ogda)
